In [22]:
import pandas as pd 
import numpy as np 
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMRegressor was fitted with feature names",
    category=UserWarning,
    module="sklearn.utils.validation"
)

In [23]:
train_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/test.csv')

# Housing Price Machine Learning - Gradient Boosting

In [5]:
#Import the requisite packages from sklearn 
#For splitting and cross validation
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV, RandomizedSearchCV

#for the pipeling 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

#evaluation metrics 
from sklearn.metrics import mean_squared_error, mean_absolute_error 

#to actually run the regressions 
#from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

In [6]:
#transform the dependent variable and get my train and test X 
y = np.log1p(train_df['SalePrice'])
X = train_df.drop(columns = ['SalePrice'])
X_test = test_df.copy()

In [7]:
#preprocess the data 
numeric_features = ['LotFrontage','LotArea','MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','1stFlrSF',
                   '2ndFlrSF','LowQualFinSF','GrLivArea','BsmtFullBath','BsmtHalfBath','FullBath','HalfBath','BedroomAbvGr',
                   'KitchenAbvGr','TotRmsAbvGrd','Fireplaces','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch',
                   '3SsnPorch','ScreenPorch','PoolArea','MiscVal']

categorical_features = ['MSSubClass','MSZoning','Street','Alley','LotShape','LandContour','Utilities','LotConfig',
                       'LandSlope', 'Neighborhood', 'Condition1','Condition2','BldgType','HouseStyle','OverallQual',
                       'OverallCond','YearBuilt','YearRemodAdd','RoofStyle','RoofMatl','Exterior1st','Exterior2nd','MasVnrType',
                       'ExterQual','ExterCond','Foundation','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
                       'Heating','HeatingQC','CentralAir','Electrical','KitchenQual','Functional','FireplaceQu','GarageType','GarageYrBlt'
                       , 'GarageFinish', 'GarageQual','GarageCond','PavedDrive','PoolQC','MiscFeature','MoSold','YrSold','SaleType',
                       'SaleCondition']

numeric_transformer = Pipeline(steps = [("imputer",SimpleImputer(strategy = "median"))
                                      ])
categorical_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "constant", fill_value = "missing")),
                                            ("to_str", FunctionTransformer(lambda x: x.astype(str))),
                                           ("onehot", OneHotEncoder(handle_unknown = "ignore"))])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
    

# XGBoost

In [10]:
from xgboost import XGBRegressor

cv = KFold(n_splits=5, shuffle=True, random_state=1234)

xgb_pipeline = Pipeline(steps = [("preprocess", preprocess), ("model", 
                                                             XGBRegressor(
                                                                 objective = "reg:squarederror",
                                                                 random_state = 1234,
                                                                 n_jobs = -1
                                                             ))])
xgb_param_dist = {
    "model__n_estimators": np.arange(500, 3500, 250),
    "model__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__subsample": [0.6, 0.75, 0.85, 1.0],
    "model__colsample_bytree": [0.6, 0.75, 0.85, 1.0],
    "model__min_child_weight": [1, 2, 5, 10],
    "model__reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
    "model__gamma": [0.0, 0.1, 0.3, 0.5],
    # optional:
    "model__reg_alpha": [0.0, 0.1, 0.5, 1.0],
}

scoring = {"rmse": "neg_root_mean_squared_error"}

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=120,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    random_state=1234,
    verbose=1
)
xgb_search.fit(X, y)
best_xgb = xgb_search.best_estimator_

print("XGB best params:", xgb_search.best_params_)
print("XGB best CV RMSE (log):", -xgb_search.best_score_)
# Optional: report CV RMSE mean±std for the tuned model
xgb_cv = cross_validate(best_xgb, X, y, cv=cv, scoring=scoring)
print("XGB 5-fold RMSE (log):", (-xgb_cv["test_rmse"]).mean(), "±", (-xgb_cv["test_rmse"]).std())


Fitting 5 folds for each of 120 candidates, totalling 600 fits
XGB best params: {'model__subsample': 1.0, 'model__reg_lambda': 2.0, 'model__reg_alpha': 0.1, 'model__n_estimators': np.int64(1750), 'model__min_child_weight': 1, 'model__max_depth': 2, 'model__learning_rate': 0.05, 'model__gamma': 0.0, 'model__colsample_bytree': 0.6}
XGB best CV RMSE (log): 0.12955710561296244
XGB 5-fold RMSE (log): 0.12955710561296244 ± 0.011041703659632312


In [12]:
# Fit on all data + Kaggle predictions
best_xgb.fit(X, y)
test_log_pred = best_xgb.predict(X_test)
test_pred = np.expm1(test_log_pred)

sub_xgb = pd.DataFrame({"Id": test_df["Id"], "SalePrice": test_pred})
sub_xgb.to_csv("/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/Submission_03022026_XGBoost.csv", index=False)


# LightGBM

In [24]:
from lightgbm import LGBMRegressor

cv = KFold(n_splits=5, shuffle=True, random_state=1234)

lgbm_pipeline = Pipeline(steps = [("preprocess", preprocess), ("model", 
                                                             LGBMRegressor(
                                                                 objective = "regression",
                                                                 random_state = 1234,
                                                                 n_jobs = -1,
                                                                 verbosity=-1,         # silences spam
                                                                 min_split_gain=0.0
                                                             ))])
lgbm_param_dist = {
    "model__n_estimators": np.arange(800, 2500, 200),
    "model__learning_rate": [ 0.02, 0.03, 0.05],

    "model__num_leaves": [31, 63, 127],          # drop 15 and 255 initially
    "model__max_depth": [-1, 3, 5, 7],           # fine

    "model__min_child_samples": [5, 10, 20, 30], # drop 40, 80 initially

    "model__subsample": [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9],

    "model__reg_lambda": [0.5, 1.0, 2.0, 5.0],
    "model__reg_alpha": [0.0, 0.1, 0.5],
}

lgbm_search = RandomizedSearchCV(
    lgbm_pipeline,
    param_distributions=lgbm_param_dist,
    n_iter=40,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
    random_state=1234,
    verbose=2
)
lgbm_search.fit(X, y)
best_lgbm = lgbm_search.best_estimator_

print("LGBM best params:", lgbm_search.best_params_)
print("LGBM best CV RMSE (log):", -lgbm_search.best_score_)
# Optional: report CV RMSE mean±std for the tuned model
scoring = {"rmse": "neg_root_mean_squared_error"}
lgbm_cv = cross_validate(best_lgbm, X, y, cv=cv, scoring=scoring)
print("LGBM 5-fold RMSE (log):", (-lgbm_cv["test_rmse"]).mean(), "±", (-lgbm_cv["test_rmse"]).std())


Fitting 5 folds for each of 40 candidates, totalling 200 fits
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.03, model__max_depth=7, model__min_child_samples=10, model__n_estimators=1400, model__num_leaves=127, model__reg_alpha=0.1, model__reg_lambda=0.5, model__subsample=0.8; total time=   3.9s
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.03, model__max_depth=7, model__min_child_samples=10, model__n_estimators=1400, model__num_leaves=127, model__reg_alpha=0.1, model__reg_lambda=0.5, model__subsample=0.8; total time=   4.0s
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.03, model__max_depth=7, model__min_child_samples=10, model__n_estimators=1400, model__num_leaves=127, model__reg_alpha=0.1, model__reg_lambda=0.5, model__subsample=0.8; total time=   4.0s
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.03, model__max_depth=7, model__min_child_samples=10, model__n_estimators=1400, model__num_leaves=127, model__reg_alpha=0.1, model_

In [15]:
X_mat = preprocess.fit_transform(X)
print(X_mat.shape)

(1460, 612)


In [25]:
best_lgbm.fit(X, y)
test_log_pred = best_lgbm.predict(X_test)
test_pred = np.expm1(test_log_pred)

sub_lgbm = pd.DataFrame({"Id": test_df["Id"], "SalePrice": test_pred})
sub_lgbm.to_csv("/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/Submission_03022026_LightGBM.csv", index=False)
